In [ ]:
# import necessary things
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

In [ ]:
dataset = 'dataset_contains_acids.csv'

In [ ]:
# read in the data
df_all = pd.read_csv(dataset)

# create a list of RDKit mol objects
smis = df_all['Substituent']
mols = [Chem.MolFromSmiles(smi) for smi in smis]
df_all['mols'] = mols

# remove the duplicated substituents (meta and para substituents are the same)
df = df_all.drop_duplicates(subset=['Substituent'])

mol_objects = df['mols'].tolist()

In [ ]:
# define acidic substructures
# smarts determined via university of hamburg's SMARTS Editor
acid1 = Chem.MolFromSmarts('[*X3](=O)[OX2H1]') #carboxylic acid and analogs
acid2 = Chem.MolFromSmarts('P(*)(O)(=O)O') #phosphonic acid
acid3 = Chem.MolFromSmarts('S(*)(O[H])(=O)=O') #sulfonic acid
acids = [acid1, acid2, acid3]

In [ ]:
excluded_pattern = Chem.MolFromSmarts('O=P(OC)(*)OC') #exclude phosphonic esters
excluded_patterns = [excluded_pattern]

In [ ]:
# Collect all matching substructures and visualize them
matching_mols = []
for acid in acids:
    matching_mols.extend([
        mol for mol in mol_objects 
        if mol.HasSubstructMatch(acid) and all(not mol.HasSubstructMatch(excluded) for excluded in excluded_patterns)
    ])

Chem.Draw.MolsToGridImage(
    matching_mols,
    molsPerRow=5,
    subImgSize=(200, 200),
    # legends=[f'Nitrile ID: {nid}' for nid in df.loc[df['mols'].isin(matching_mols), 'para_nitrile_id']],
    maxMols=len(matching_mols)
)

In [ ]:
# Create a dataframe containing identified acids
acids_df = df[df['mols'].isin(matching_mols)].copy()
acids_df.reset_index(drop=True, inplace=True)
acids_df = acids_df.drop(columns=['mols'])

acids_df.head()

In [ ]:
# remove acids from the dataset
df_all = df_all[~df_all['mols'].isin(matching_mols)].copy()
df_all.to_csv('hammett_with_smiles.csv')

In [ ]:
# import necessary packages for visualizing the acids
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import seaborn as sns

# define colors used in this work
paton_hot_pink = '#d00d44'
rosewood_plum = '#9B6B8E'     
rosewood_sage = '#7BA882'     
rosewood_gold = '#D4A574'       
rosewood_slate = '#5B7FA6'     
rosewood_rose = '#C9858C'        
rosewood_charcoal = '#4A4A4A'    
rosewood_cream = '#F5EDE0' 
tol_grey = '#BBBBBB'

# define plotting parameters
sns.set(context='talk', style='ticks',
        color_codes=True, rc={'legend.frameon': False})
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 300
plt.rcParams["font.family"] = 'Avenir'
plt.rcParams.update({'font.size': 24})
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# visualize the data without acids

# read in the data
full_dataset = 'dataset_contains_acids.csv'
all = pd.read_csv(full_dataset)
df = all

# remove H from the para substituents so it is not double-counted
df = df[df['nitrile_id'] != 'para_nitrile_032']

# perform linear regression on the full dataset
a = df['cm5_charge'].values.reshape(-1, 1)
b = df['hammett_parameter'].values.reshape(-1, 1)

regr = linear_model.LinearRegression()
regr.fit(a, b)

slope = regr.coef_[0][0]
intercept = regr.intercept_[0]
print(f"Equation of the line (all): y = {slope:.2f}x + {intercept:.2f}")

# predict hammett parameters when including acids in the regression
df['predicted_hammett_with_acids'] = (slope * df['cm5_charge'] + intercept)

# perform a linear regression and predict hammett parameters without acids
df_acids = pd.read_csv('filtered_acids.csv')

acids = df['Substituent'].isin(df_acids['Substituent']) 

c = a[~acids]
d = b[~acids]
regr_sans_acids = linear_model.LinearRegression()
regr_sans_acids.fit(c, d)

slope_sans_acids = regr_sans_acids.coef_[0][0]
intercept_sans_acids = regr_sans_acids.intercept_[0]
print(f"Equation of the line (sans acids): y = {slope_sans_acids:.2f}x + {intercept_sans_acids:.2f}")

df['predicted_hammett_without_acids'] = (slope_sans_acids * df['cm5_charge'] + intercept_sans_acids)

# plot the data
x = df['hammett_parameter'].values.reshape(-1, 1)
y = df['predicted_hammett_with_acids'].values.reshape(-1, 1)
z = df['predicted_hammett_without_acids'].values.reshape(-1, 1)

regrs = linear_model.LinearRegression()
regrs.fit(x, y)
r_value = r2_score(y, regrs.predict(x))

mae = mean_absolute_error(y, x)

print(f"R^2 with acids: {r_value:.2f}")
print(f"MAE with acids: {mae:.2f}")

regr_sans_acids = linear_model.LinearRegression()
regr_sans_acids.fit(x[~acids], z[~acids])
r_value_sans_acids = r2_score(z[~acids], regrs.predict(x[~acids]))

mae_sans_acids = mean_absolute_error(z[~acids], x[~acids])

plt.axline((0, 0), slope=1, color=tol_grey, linestyle="--", zorder=1)
plt.scatter(x[~acids],y[~acids], color=rosewood_slate, alpha=0.2, s=35, edgecolors=rosewood_slate, zorder=0)
plt.scatter(x[acids], y[acids], alpha=0.7, color=paton_hot_pink, edgecolors=paton_hot_pink, marker='^', s=35, zorder=2)
plt.xlabel('Experimental Hammett Parameter', fontsize=16)
plt.ylabel('Predicted Hammett Parameter', fontsize=16)
plt.xlim(-0.75, 1.5)
plt.xticks(np.linspace(-0.5, 1.5, 5), [f"{tick:.1f}" for tick in np.linspace(-0.5, 1.5, 5)])
plt.ylim(-0.75, 1.5)
plt.yticks(np.linspace(-0.5, 1.5, 5), [f"{tick:.1f}" for tick in np.linspace(-0.5, 1.5, 5)])
plt.annotate(f'R$^2$ = {"{:.2f}".format(r_value_sans_acids)}', xy=(0.05,0.90), xycoords='axes fraction', fontsize=16)
plt.annotate(f'MAE = {"{:.2f}".format(mae_sans_acids)}', xy=(0.05,0.83), xycoords='axes fraction', fontsize=16)

# plt.savefig('acids-fig.svg', bbox_inches='tight', dpi=400)

plt.show()